In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

RANDOM_STATE = 42

DATA_PATH = Path('cleaned_data.csv')

df = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
print('Shape:', df.shape)
display(df.head())

In [ ]:
df.columns = df.columns.str.strip().str.upper().str.replace(r"[^A-Z0-9_]+", "_", regex=True).str.strip("_")

In [ ]:
df.info()

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
print(f"Memory before: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.dtypes.value_counts()

In [ ]:
df.loc[
    df["APP_DATE"].notnull(),
    ["APP_DATE"]
].head(5) 

In [ ]:
df["APP_DATE"].dtypes 

In [ ]:
df["APP_DATE"].dtype.name

In [ ]:
df.head()

In [ ]:
# Dates stored as YYYYMMDD integers -> real datetime
for col in ['DOB', 'OPENING_DATE']:
    df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')

# Dates already stored as "YYYY-MM-DD" strings -> real datetime (still parsed as object until converted)
for col in ['APP_DATE', 'MATURITY_DATE', 'REPAY_ST_DATE', 'BUSINESS_DATE']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

df[['DOB', 'OPENING_DATE', 'APP_DATE', 'MATURITY_DATE', 'REPAY_ST_DATE', 'BUSINESS_DATE']].dtypes

In [ ]:
df.head()

### Maturity filter -- exclude right-censored (too-recent) originations

`BUSINESS_DATE` is a fixed snapshot (2026-07-31) for every row, so `IS_NPL` as of that date is
only a trustworthy label for loans old enough to have had a fair chance to reach the NPL
threshold. A vintage-curve check (3-month rolling NPL rate by `APP_DATE` month) shows the rate
rising to a peak around Jan/Feb 2024, then declining every single month -- an unbroken 30-month
monotonic decline down to 0% by 2026-03. That pattern is the signature of loans too young to
have defaulted yet, not genuinely improving risk. Loans originated after the peak are excluded
from modeling below.

In [ ]:
import matplotlib.pyplot as plt

MATURITY_CUTOFF = pd.Timestamp('2024-02-29')

vintage = df.set_index('APP_DATE')['IS_NPL'].resample('M').agg(['count', 'mean'])
vintage['rolling_rate'] = vintage['mean'].rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vintage.index, vintage['mean'], alpha=0.4, label='Monthly NPL rate')
ax.plot(vintage.index, vintage['rolling_rate'], color='C1', label='3-month rolling rate')
ax.axvline(MATURITY_CUTOFF, color='red', linestyle='--', label=f'Maturity cutoff ({MATURITY_CUTOFF.date()})')
ax.set_xlabel('Loan origination month (APP_DATE)')
ax.set_ylabel('IS_NPL rate')
ax.set_title('NPL Rate by Vintage Month -- Right-Censoring Check')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Apply the maturity filter: keep only originations old enough to have had a fair
# chance to reach the NPL threshold as of the BUSINESS_DATE snapshot.
before_rows = len(df)
before_rate = df['IS_NPL'].mean()

df = df[df['APP_DATE'] <= MATURITY_CUTOFF].reset_index(drop=True)

after_rows = len(df)
after_rate = df['IS_NPL'].mean()

print(f'Maturity cutoff: originated on or before {MATURITY_CUTOFF.date()}')
print(f'Rows before filter: {before_rows}  (NPL rate {before_rate:.4%})')
print(f'Rows after filter:  {after_rows}  (NPL rate {after_rate:.4%})')
print(f'Dropped {before_rows - after_rows} immature/right-censored rows ({(before_rows - after_rows) / before_rows:.2%})')


In [ ]:
df.info()

In [ ]:
# Binary Y/N columns -> bool
yn_cols = ['STAFF_OFFICIAL', 'LONG_TERM', 'PMS_NEWENQ', 'PMS_PASSDUE', 'PMS_RESTRUCTURE']
df[yn_cols] = df[yn_cols].apply(lambda s: s.map({'Y': True, 'N': False}))
df[yn_cols].dtypes

In [ ]:
df[["STAFF_OFFICIAL", "LONG_TERM", "PMS_NEWENQ", "PMS_PASSDUE", "PMS_RESTRUCTURE"]].head()

In [ ]:
df[["STAFF_OFFICIAL", "LONG_TERM", "PMS_NEWENQ", "PMS_PASSDUE", "PMS_RESTRUCTURE"]].info()

#### standardizes column names — cleaning up messy/inconsistent header
1. remove leading/trailing whitespace
2. force uppercase
3. replace any non-alphanumeric run with "_"
4. trim leading/trailing underscores

In [ ]:
# Row-value standardization: strip stray whitespace and unify casing BEFORE
# casting to category, otherwise inconsistent text ('Boeng Kak' vs 'Boeng kak')
# locks in as separate categories. Must run before the category-dtype cell below.
cat_cols = [
    'BRANCH_CODE', 'BRANCH_CODE_1', 'CUST_TYPE', 'GENDER', 'MARITAL_STATUS',
    'TARGET_DESC', 'SECTOR_DESC', 'INDUSTRY_DESC', 'NATIONALITY',
    'OCCUP_TYPE_DESC', 'OCCUP_DET_DESC', 'PROVINCE', 'DISTRICT', 'COMMUNE',
    'VILLAGE', 'AMK_BIRTH_PLACE_DESC', 'POOR_STATUS', 'L_AML_RISK_RATE',
    'L_AML_RISK_RATE_DESC', 'ONBOARD_CHANNEL', 'URBAN', 'PR_REPAY_FREQ',
    'PRODUCT_TYPE', 'OPERATION', 'LOAN_REFER_BY', 'LOAN_STATUS', 'INC_EXP_CCY',
    'APPLN_SOURCE', 'PURPOSE_SECTOR', 'BUSINESS_SECTOR', 'CURRENCY',
]

before_nunique = {c: df[c].nunique() for c in cat_cols}
df[cat_cols] = df[cat_cols].apply(lambda s: s.str.strip().str.title())
after_nunique = {c: df[c].nunique() for c in cat_cols}

changed = {c: (before_nunique[c], after_nunique[c]) for c in cat_cols if before_nunique[c] != after_nunique[c]}
print("Columns where duplicate categories were merged (before -> after):")
print(changed)

In [ ]:
# Low-cardinality object columns -> category (main memory win)
cat_cols = [
    'BRANCH_CODE', 'BRANCH_CODE_1', 'CUST_TYPE', 'GENDER', 'MARITAL_STATUS',
    'TARGET_DESC', 'SECTOR_DESC', 'INDUSTRY_DESC', 'NATIONALITY',
    'OCCUP_TYPE_DESC', 'OCCUP_DET_DESC', 'PROVINCE', 'DISTRICT', 'COMMUNE',
    'VILLAGE', 'AMK_BIRTH_PLACE_DESC', 'POOR_STATUS', 'L_AML_RISK_RATE',
    'L_AML_RISK_RATE_DESC', 'ONBOARD_CHANNEL', 'URBAN', 'PR_REPAY_FREQ',
    'PRODUCT_TYPE', 'OPERATION', 'LOAN_REFER_BY', 'LOAN_STATUS', 'INC_EXP_CCY',
    'APPLN_SOURCE', 'PURPOSE_SECTOR', 'BUSINESS_SECTOR', 'CURRENCY',
]
df[cat_cols] = df[cat_cols].astype('category')
df[cat_cols].dtypes

In [ ]:
# M0_PSC..M12_PSC: monthly delinquency-bucket score (0-13), keep numeric/ordinal,
# not category -- higher values mean more overdue, and that ordering matters downstream.
# Unify dtype: M0_PSC currently int64, M1-M12 are float64 -> make them all consistent int8.
psc_cols = [f'M{i}_PSC' for i in range(13)]
assert df[psc_cols].isna().sum().sum() == 0
df[psc_cols] = df[psc_cols].astype('int8')
df[psc_cols].dtypes

In [ ]:
df["RESIDENCE_N_Y"].unique()

In [ ]:
df["RATE_TYPE"].unique()

In [ ]:
# Zero-variance columns carry no information -> drop
constant_cols = ['RESIDENCE_N_Y', 'RATE_TYPE']
print({c: df[c].nunique() for c in constant_cols})
df = df.drop(columns=constant_cols)

# Target -> compact int
df['IS_NPL'] = df['IS_NPL'].astype('int8')

In [ ]:
df.shape

In [ ]:
df["UP_FRONT_FEE_VC"].isnull().sum() / len (df) * 100 

In [ ]:
df.loc[
    df["UP_FRONT_FEE_VC"].notnull(),
    ["UP_FRONT_FEE_VC"]
].head()

In [ ]:
# MB_LOCAL_TRANSFER_CNT is ~93% empty -> missing almost certainly means "no local transfers", fill 0.
df['UP_FRONT_FEE_VC'] = df['UP_FRONT_FEE_VC'].fillna(0)
df['MB_LOCAL_TRANSFER_CNT'] = df['MB_LOCAL_TRANSFER_CNT'].fillna(0)

assert df.isna().sum().sum() == 0, "unexpected nulls remain after missing-data handling"
print("Total nulls remaining:", df.isna().sum().sum())

In [ ]:
# Define the ML-ready feature scope.
# Kept in df for traceability/joins, but excluded from modeling:
#   - identifiers: not predictive, and using them as features would leak row identity
#   - raw datetime64 columns: tree libraries (LightGBM/XGBoost/CatBoost) don't accept
#     datetime64 directly -- these need explicit feature engineering (e.g. tenure in
#     days, age at application) as a separate step before they can be used as features.
ID_COLS = ['CUSTOMER_ID', 'ACCOUNT_NUM', 'ACCOUNT_OFFICER', 'TEL_MOBILE', 'BRANCH_CODE_1']
DATE_COLS = ['DOB', 'OPENING_DATE', 'APP_DATE', 'MATURITY_DATE', 'REPAY_ST_DATE', 'BUSINESS_DATE']
TARGET_COL = 'IS_NPL'

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + DATE_COLS + [TARGET_COL]]

# Tree-based models accept numeric/bool/category dtypes directly -- object or
# datetime64 sneaking into FEATURE_COLS means something wasn't standardized.
non_ml_dtypes = df[FEATURE_COLS].select_dtypes(include=['object', 'datetime64[ns]']).columns.tolist()
assert not non_ml_dtypes, f"non-ML-ready dtypes still in FEATURE_COLS: {non_ml_dtypes}"
assert df[FEATURE_COLS].isna().sum().sum() == 0, "missing values remain in FEATURE_COLS"

print(f"n features: {len(FEATURE_COLS)}")
df[FEATURE_COLS].dtypes.apply(lambda d: d.name).value_counts()

In [ ]:
# Final check: confirm standardization and memory savings
print(f"Memory after standardization: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.info()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.head()

#### Priority 1 — confirmed data errors, check these first:
- AGE
- TOTAL_INCOME
- TOTAL_EXPENSE
- AVG_INCOME
- FAMILY_MEMBER
- CASA_BALANCE_USD

In [ ]:
df['AGE'].describe()

In [ ]:
df[df['AGE'] == 150]

In [ ]:
df["AGE"].value_counts().sort_index()

In [ ]:
df = df[(df['AGE'] >= 18) & (df['AGE'] <= 70)]

In [ ]:
df['AGE'].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["AGE"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")

# Boxplot
sns.boxplot(x=df["AGE"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")

# Scatter plot (row index vs AGE) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["AGE"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs AGE)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("AGE")

plt.tight_layout()
plt.show()

In [ ]:
df["AGE"].value_counts(dropna=False).sort_index()

In [ ]:
df.head()

In [ ]:
df['BRANCH_CODE'] = df['BRANCH_CODE'].str.upper()
df['NATIONALITY'] = df['NATIONALITY'].str.upper()
df['L_AML_RISK_RATE_DESC'] = df['L_AML_RISK_RATE_DESC'].str.upper()
df['POOR_STATUS'] = df['POOR_STATUS'].str.upper()
df["POOR_STATUS"] = df['POOR_STATUS'].str.upper()
df["LOAN_REFER_BY"] = df['LOAN_REFER_BY'].str.upper()
df["INC_EXP_CCY"] = df['INC_EXP_CCY'].str.upper()
df["CURRENCY"] = df['CURRENCY'].str.upper()
df["BRANCH_CODE_1"] = df['BRANCH_CODE_1'].str.upper()

In [ ]:
df["TOTAL_INCOME"].describe()

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df["TOTAL_INCOME"].describe()

In [ ]:
df.head()

In [ ]:
# LOAN_AMOUNT is stored in native currency (see CURRENCY column), not converted to USD like the *_USD columns.
# Approximate fixed rates (Cambodia/AMK context) -- verify/replace with actual daily rates if precision matters.
FX_TO_USD = {'USD': 1, 'KHR': 4000, 'THB': 40}

# CURRENCY is 'category' dtype -> cast to str first, otherwise .map() returns a categorical that can't divide
df['TOTAL_INCOME_USD'] = df['TOTAL_INCOME'] / df['INC_EXP_CCY'].astype(str).map(FX_TO_USD)
df['TOTAL_INCOME_USD'].isnull().sum()  # should be 0 if every CURRENCY value is in the map above

In [ ]:
df['TOTAL_INCOME_USD'].head()

In [ ]:
df["TOTAL_INCOME_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Histogram
sns.histplot(df["TOTAL_INCOME_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")

# Boxplot
sns.boxplot(x=df["TOTAL_INCOME_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")


plt.tight_layout()
plt.show()

In [ ]:
df["TOTAL_EXPENSE"].describe()

In [ ]:
# LOAN_AMOUNT is stored in native currency (see CURRENCY column), not converted to USD like the *_USD columns.
# Approximate fixed rates (Cambodia/AMK context) -- verify/replace with actual daily rates if precision matters.
FX_TO_USD = {'USD': 1, 'KHR': 4000, 'THB': 40}

# CURRENCY is 'category' dtype -> cast to str first, otherwise .map() returns a categorical that can't divide
df['TOTAL_EXPENSE_USD'] = df['TOTAL_EXPENSE'] / df['INC_EXP_CCY'].astype(str).map(FX_TO_USD)
df['TOTAL_EXPENSE_USD'].isnull().sum()  # should be 0 if every CURRENCY value is in the map above

In [ ]:
df["TOTAL_EXPENSE_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["TOTAL_EXPENSE_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("TOTAL_EXPENSE_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["TOTAL_EXPENSE_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("TOTAL_EXPENSE_USD")
axes[1].set_ylabel("TOTAL_EXPENSE_USD")
# Scatter plot (row index vs TOTAL_EXPENSE_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["TOTAL_EXPENSE_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs TOTAL_EXPENSE_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("TOTAL_EXPENSE_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["AVG_INCOME"].describe()

In [ ]:
# LOAN_AMOUNT is stored in native currency (see CURRENCY column), not converted to USD like the *_USD columns.
# Approximate fixed rates (Cambodia/AMK context) -- verify/replace with actual daily rates if precision matters.
FX_TO_USD = {'USD': 1, 'KHR': 4000, 'THB': 40}

# CURRENCY is 'category' dtype -> cast to str first, otherwise .map() returns a categorical that can't divide
df['AVG_INCOME_USD'] = df['AVG_INCOME'] / df['INC_EXP_CCY'].astype(str).map(FX_TO_USD)
df['AVG_INCOME_USD'].isnull().sum()  # should be 0 if every CURRENCY value is in the map above

In [ ]:
df["AVG_INCOME_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["AVG_INCOME_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("AVG_INCOME_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["AVG_INCOME_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("AVG_INCOME_USD")
axes[1].set_ylabel("AVG_INCOME_USD")
# Scatter plot (row index vs AVG_INCOME_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["AVG_INCOME_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs AVG_INCOME_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("AVG_INCOME_USD")

plt.tight_layout()
plt.show()

In [ ]:
df['FAMILY_MEMBER'].value_counts().sort_index(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count FAMILY_MEMBER
family_counts = df['FAMILY_MEMBER'].value_counts().sort_index()

# Plot
plt.figure(figsize=(12, 6))

sns.barplot(
    x=family_counts.index,
    y=family_counts.values,
    color='steelblue'
)

plt.title('Distribution of Family Members', fontsize=16)
plt.xlabel('Number of Family Members', fontsize=12)
plt.ylabel('Number of Records', fontsize=12)

plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
df = df[(df['FAMILY_MEMBER'] >= 1) & (df['FAMILY_MEMBER'] <= 15)]

In [ ]:
df['FAMILY_MEMBER'].value_counts().sort_index(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count FAMILY_MEMBER
family_counts = df['FAMILY_MEMBER'].value_counts().sort_index()

# Plot
plt.figure(figsize=(12, 6))

sns.barplot(
    x=family_counts.index,
    y=family_counts.values,
    color='steelblue'
)

plt.title('Distribution of Family Members', fontsize=16)
plt.xlabel('Number of Family Members', fontsize=12)
plt.ylabel('Number of Records', fontsize=12)

plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
df["CASA_BALANCE_USD"].describe()

In [ ]:
df.loc[
    df['CASA_BALANCE_USD'] < 0,
    'CASA_BALANCE_USD'
].head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

col = "CASA_BALANCE_USD"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(
    x=df[col].dropna(),
    ax=axes[0]
)
axes[0].set_xscale("symlog", linthresh=1)
axes[0].set_title("CASA Balance Boxplot")
axes[0].set_xlabel("CASA Balance (USD)")
axes[0].grid(alpha=0.3)

# Scatter plot
axes[1].scatter(
    df.index,
    df[col],
    alpha=0.25,
    s=8
)
axes[1].set_yscale("symlog", linthresh=1)
axes[1].set_title("CASA Balance by Row")
axes[1].set_xlabel("Row Index")
axes[1].set_ylabel("CASA Balance (USD)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


#### Priority 2 — heavy-tailed amounts, worth capping:
- LOAN_AMOUNT_USD
- BALANCE_USD
- CONTRACTUAL_BALANCE_USD
- TERM_BALANCE_USD

In [ ]:
df.head()

In [ ]:
df["LOAN_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["LOAN_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("LOAN_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["LOAN_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("LOAN_AMOUNT_USD")
axes[1].set_ylabel("LOAN_AMOUNT_USD")
# Scatter plot (row index vs LOAN_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["LOAN_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs LOAN_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("LOAN_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["BALANCE_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["BALANCE_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("BALANCE_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["BALANCE_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("BALANCE_USD")
axes[1].set_ylabel("BALANCE_USD")
# Scatter plot (row index vs BALANCE_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["BALANCE_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs BALANCE_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("BALANCE_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["CONTRACTUAL_BALANCE_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CONTRACTUAL_BALANCE_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CONTRACTUAL_BALANCE_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CONTRACTUAL_BALANCE_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CONTRACTUAL_BALANCE_USD")
axes[1].set_ylabel("CONTRACTUAL_BALANCE_USD")
# Scatter plot (row index vs CONTRACTUAL_BALANCE_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CONTRACTUAL_BALANCE_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CONTRACTUAL_BALANCE_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CONTRACTUAL_BALANCE_USD")

plt.tight_layout()
plt.show()

#### Priority 3 — zero-inflated transaction columns, lower urgency:

- CASA_DEBIT_CNT
- CASA_CREDIT_CNT
- CASA_DEBIT_AMOUNT_USD
- CASA_CREDIT_AMOUNT_USD
- MB_LOCAL_TRANSFER_CNT
- MB_LOCAL_TRANSFER_AMOUNT_USD
- MB_CASH_WITHDRAWAL_CNT
- MB_CASH_WITHDRAWAL_AMOUNT_USD
- MB_PAYMENT_CNT
- MB_PAYMENT_AMOUNT_USD
- BKI_TRN_COUNT
- BKI_TRN_AMOUNT_USD
- CS_CASH_DEPOSIT_CNT
- CS_CASH_DEPOSIT_AMOUNT_USD
- CS_PURCHASE_CNT
- CS_PURCHASE_AMOUNT_USD


CASA_CREDIT_AMOUNT_USD
MB_LOCAL_TRANSFER_CNT

In [ ]:
df["CASA_DEBIT_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CASA_DEBIT_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CASA_DEBIT_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CASA_DEBIT_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CASA_DEBIT_CNT")
axes[1].set_ylabel("CASA_DEBIT_CNT")
# Scatter plot (row index vs CASA_DEBIT_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CASA_DEBIT_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CASA_DEBIT_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CASA_DEBIT_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["CASA_CREDIT_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CASA_CREDIT_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CASA_CREDIT_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CASA_CREDIT_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CASA_CREDIT_CNT")
axes[1].set_ylabel("CASA_CREDIT_CNT")
# Scatter plot (row index vs CASA_CREDIT_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CASA_CREDIT_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CASA_CREDIT_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CASA_CREDIT_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["CASA_DEBIT_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CASA_DEBIT_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CASA_DEBIT_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CASA_DEBIT_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CASA_DEBIT_AMOUNT_USD")
axes[1].set_ylabel("CASA_DEBIT_AMOUNT_USD")
# Scatter plot (row index vs CASA_DEBIT_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CASA_DEBIT_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CASA_DEBIT_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CASA_DEBIT_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["CASA_CREDIT_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CASA_CREDIT_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CASA_CREDIT_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CASA_CREDIT_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CASA_CREDIT_AMOUNT_USD")
axes[1].set_ylabel("CASA_CREDIT_AMOUNT_USD")
# Scatter plot (row index vs CASA_CREDIT_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CASA_CREDIT_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CASA_CREDIT_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CASA_CREDIT_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["MB_LOCAL_TRANSFER_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["MB_LOCAL_TRANSFER_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("MB_LOCAL_TRANSFER_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["MB_LOCAL_TRANSFER_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("MB_LOCAL_TRANSFER_CNT")
axes[1].set_ylabel("MB_LOCAL_TRANSFER_CNT")
# Scatter plot (row index vs MB_LOCAL_TRANSFER_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["MB_LOCAL_TRANSFER_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs MB_LOCAL_TRANSFER_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("MB_LOCAL_TRANSFER_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["MB_LOCAL_TRANSFER_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["MB_LOCAL_TRANSFER_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("MB_LOCAL_TRANSFER_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["MB_LOCAL_TRANSFER_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("MB_LOCAL_TRANSFER_AMOUNT_USD")
axes[1].set_ylabel("MB_LOCAL_TRANSFER_AMOUNT_USD")
# Scatter plot (row index vs MB_LOCAL_TRANSFER_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["MB_LOCAL_TRANSFER_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs MB_LOCAL_TRANSFER_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("MB_LOCAL_TRANSFER_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["MB_CASH_WITHDRAWAL_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["MB_CASH_WITHDRAWAL_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("MB_CASH_WITHDRAWAL_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["MB_CASH_WITHDRAWAL_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("MB_CASH_WITHDRAWAL_CNT")
axes[1].set_ylabel("MB_CASH_WITHDRAWAL_CNT")
# Scatter plot (row index vs MB_CASH_WITHDRAWAL_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["MB_CASH_WITHDRAWAL_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs MB_CASH_WITHDRAWAL_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("MB_CASH_WITHDRAWAL_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["MB_CASH_WITHDRAWAL_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["MB_CASH_WITHDRAWAL_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("MB_CASH_WITHDRAWAL_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["MB_CASH_WITHDRAWAL_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("MB_CASH_WITHDRAWAL_AMOUNT_USD")
axes[1].set_ylabel("MB_CASH_WITHDRAWAL_AMOUNT_USD")
# Scatter plot (row index vs MB_CASH_WITHDRAWAL_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["MB_CASH_WITHDRAWAL_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs MB_CASH_WITHDRAWAL_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("MB_CASH_WITHDRAWAL_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["MB_PAYMENT_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["MB_PAYMENT_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("MB_PAYMENT_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["MB_PAYMENT_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("MB_PAYMENT_CNT")
axes[1].set_ylabel("MB_PAYMENT_CNT")
# Scatter plot (row index vs MB_PAYMENT_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["MB_PAYMENT_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs MB_PAYMENT_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("MB_PAYMENT_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["MB_PAYMENT_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["MB_PAYMENT_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("MB_PAYMENT_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["MB_PAYMENT_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("MB_PAYMENT_AMOUNT_USD")
axes[1].set_ylabel("MB_PAYMENT_AMOUNT_USD")
# Scatter plot (row index vs MB_PAYMENT_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["MB_PAYMENT_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs MB_PAYMENT_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("MB_PAYMENT_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["BKI_TRN_COUNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["BKI_TRN_COUNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("BKI_TRN_COUNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["BKI_TRN_COUNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("BKI_TRN_COUNT")
axes[1].set_ylabel("BKI_TRN_COUNT")
# Scatter plot (row index vs BKI_TRN_COUNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["BKI_TRN_COUNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs BKI_TRN_COUNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("BKI_TRN_COUNT")

plt.tight_layout()
plt.show()

In [ ]:
df["BKI_TRN_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["BKI_TRN_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("BKI_TRN_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["BKI_TRN_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("BKI_TRN_AMOUNT_USD")
axes[1].set_ylabel("BKI_TRN_AMOUNT_USD")
# Scatter plot (row index vs BKI_TRN_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["BKI_TRN_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs BKI_TRN_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("BKI_TRN_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["CS_CASH_DEPOSIT_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CS_CASH_DEPOSIT_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CS_CASH_DEPOSIT_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CS_CASH_DEPOSIT_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CS_CASH_DEPOSIT_CNT")
axes[1].set_ylabel("CS_CASH_DEPOSIT_CNT")
# Scatter plot (row index vs CS_CASH_DEPOSIT_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CS_CASH_DEPOSIT_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CS_CASH_DEPOSIT_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CS_CASH_DEPOSIT_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["CS_CASH_DEPOSIT_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CS_CASH_DEPOSIT_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CS_CASH_DEPOSIT_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CS_CASH_DEPOSIT_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CS_CASH_DEPOSIT_AMOUNT_USD")
axes[1].set_ylabel("CS_CASH_DEPOSIT_AMOUNT_USD")
# Scatter plot (row index vs CS_CASH_DEPOSIT_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CS_CASH_DEPOSIT_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CS_CASH_DEPOSIT_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CS_CASH_DEPOSIT_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["CS_PURCHASE_CNT"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CS_PURCHASE_CNT"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CS_PURCHASE_CNT")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CS_PURCHASE_CNT"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CS_PURCHASE_CNT")
axes[1].set_ylabel("CS_PURCHASE_CNT")
# Scatter plot (row index vs CS_PURCHASE_CNT) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CS_PURCHASE_CNT"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CS_PURCHASE_CNT)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CS_PURCHASE_CNT")

plt.tight_layout()
plt.show()

In [ ]:
df["CS_PURCHASE_AMOUNT_USD"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["CS_PURCHASE_AMOUNT_USD"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("CS_PURCHASE_AMOUNT_USD")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["CS_PURCHASE_AMOUNT_USD"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("CS_PURCHASE_AMOUNT_USD")
axes[1].set_ylabel("CS_PURCHASE_AMOUNT_USD")
# Scatter plot (row index vs CS_PURCHASE_AMOUNT_USD) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["CS_PURCHASE_AMOUNT_USD"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs CS_PURCHASE_AMOUNT_USD)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("CS_PURCHASE_AMOUNT_USD")

plt.tight_layout()
plt.show()

In [ ]:
df["PROFESSION"].unique()

In [ ]:
df["PROFESSION"].value_counts().sort_index(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 4))

# Histogram
sns.histplot(df["PROFESSION"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("PROFESSION")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["PROFESSION"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("PROFESSION")
axes[1].set_ylabel("PROFESSION")

plt.tight_layout()
plt.show()

In [ ]:
df["ACTIVE_MEMBER"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["ACTIVE_MEMBER"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("ACTIVE_MEMBER")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["ACTIVE_MEMBER"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("ACTIVE_MEMBER")
axes[1].set_ylabel("ACTIVE_MEMBER")
# Scatter plot (row index vs ACTIVE_MEMBER) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["ACTIVE_MEMBER"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs ACTIVE_MEMBER)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("ACTIVE_MEMBER")

plt.tight_layout()
plt.show()

In [ ]:
df["LOAN_CYCLE"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
sns.histplot(df["LOAN_CYCLE"], bins=50, ax=axes[0])
axes[0].set_title("Distribution")
axes[0].set_xlabel("LOAN_CYCLE")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df["LOAN_CYCLE"], ax=axes[1])
axes[1].set_title("Boxplot (outliers)")
axes[1].set_xlabel("LOAN_CYCLE")
axes[1].set_ylabel("LOAN_CYCLE")
# Scatter plot (row index vs LOAN_CYCLE) -> makes extreme individual values easy to spot
axes[2].scatter(df.index, df["LOAN_CYCLE"], alpha=0.3, s=10)
axes[2].set_title("Scatter (row index vs LOAN_CYCLE)")
axes[2].set_xlabel("Row Index")
axes[2].set_ylabel("LOAN_CYCLE")

plt.tight_layout()
plt.show()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
def move_loan_column(df):
    cols = list(df.columns)
    if "TOTAL_INCOME_USD" in cols and "TOTAL_INCOME" in cols and "INC_EXP_CCY" in cols:
        cols.remove("TOTAL_INCOME")
        cols.remove("TOTAL_INCOME_USD")  # remove old position before reinserting, otherwise it duplicates
        cols.insert(cols.index("INC_EXP_CCY") + 1, "TOTAL_INCOME_USD")
        return df[cols]
    return df  # Returns unchanged if columns don't exist

# Call the function
df = move_loan_column(df)

In [ ]:
df.head()

In [ ]:
def move_loan_column(df):
    cols = list(df.columns)
    if "TOTAL_EXPENSE_USD" in cols and "TOTAL_EXPENSE" in cols and "INC_EXP_CCY" in cols:
        cols.remove("TOTAL_EXPENSE")
        cols.remove("TOTAL_EXPENSE_USD")  # remove old position before reinserting, otherwise it duplicates
        cols.insert(cols.index("INC_EXP_CCY") + 1, "TOTAL_EXPENSE_USD")
        return df[cols]
    return df  # Returns unchanged if columns don't exist

# Call the function
df = move_loan_column(df)

In [ ]:
def move_loan_column(df):
    cols = list(df.columns)
    if "AVG_INCOME_USD" in cols and "AVG_INCOME" in cols and "INC_EXP_CCY" in cols:
        cols.remove("AVG_INCOME")
        cols.remove("AVG_INCOME_USD")  # remove old position before reinserting, otherwise it duplicates
        cols.insert(cols.index("INC_EXP_CCY") + 1, "AVG_INCOME_USD")
        return df[cols]
    return df  # Returns unchanged if columns don't exist

# Call the function
df = move_loan_column(df)

In [ ]:
df.head()

In [ ]:
df.to_csv('ready_for_featrue_selection.csv', index=False)

In [ ]:
df.info()

In [ ]:
df.shape